In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from mlxtend.frequent_patterns import apriori, association_rules
import datetime as dt
import os

print("--- Step 1: Data Loading & Cleaning ---")
# Read data directly from your local tracking folder path
df = pd.read_csv('../data/data.csv', encoding='ISO-8859-1')

# Clean missing records and drop negative transactional returns
df.dropna(subset=['CustomerID'], inplace=True)
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['CustomerID'] = df['CustomerID'].astype(int)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalRevenue'] = df['Quantity'] * df['UnitPrice']
print(f"Cleaned dataset shape: {df.shape}\n")

print("--- Step 2: Exploratory Data Analysis (EDA) ---")
plt.figure(figsize=(10, 4))
df.groupby('Country')['TotalRevenue'].sum().sort_values(ascending=False).head(5).plot(kind='bar', color='darkturquoise')
plt.title('Top 5 Countries by Total Revenue')
plt.ylabel('Revenue')
plt.xticks(rotation=45)
plt.show() # This renders your graph instantly inside your notebook panel!

print("--- Step 3: RFM Customer Segmentation (K-Means) ---")
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalRevenue': 'sum'
}).rename(columns={'InvoiceDate': 'Recency', 'InvoiceNo': 'Frequency', 'TotalRevenue': 'Monetary'})

# Transform skewed metric variations log-arithmetically
rfm_log = np.log1p(rfm)
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

# Cluster into three distinct buyer types
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)
print("\nGenerated Cluster Evaluation Matrix:")
print(rfm.groupby('Cluster').mean().round(1))

print("\n--- Step 4: Product Association Rule Discovery ---")
basket = (df[df['Country'] =="France"]
          .groupby(['InvoiceNo', 'Description'])['Quantity']
          .sum().unstack().reset_index().fillna(0)
          .set_index('InvoiceNo'))

def encode_units(x):
    return 1 if x >= 1 else 0
basket_sets = basket.applymap(encode_units)

frequent_itemsets = apriori(basket_sets, min_support=0.05, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
rules = rules.sort_values(by='lift', ascending=False).head(5)
print("\nTop Strongest Product Cross-Selling Pairs:")
display(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])
